## Tworzenie projektu.

In [ ]:
import findspark
# Projekt: Amudsen.
# Wersja: 1.5.0.

import os

#from IPython import display

# import dbutils
#import sys

# JAVA_HOME musi być ustawione przed importem biblioteki pyspark.
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
#findspark.init("/opt/spark")

# Konfiguracja projektu z importowaniem wymaganych bibliotek.
from pyspark.sql import *
#from pyspark.rdd import *
from delta import *

#from pyspark.sql.types import IntegerType

builder = (
    SparkSession
        .builder
        .master("local[*]") \
        .appName("Amudsen")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

# Utworzenie sesji.
session = configure_spark_with_delta_pip(builder).getOrCreate()

# Konfiguracja dodatkowych parametrów sesji po jej utworzeniu.
session.conf.set("spark.sql.shuffle.partitions", "20")

In [ ]:
# Warunkowe usunięcie tabel.
'''
session.sql("DROP TABLE IF EXISTS country")
session.sql("DROP TABLE IF EXISTS glacier")
session.sql("DROP TABLE IF EXISTS peak")
session.sql("DROP TABLE IF EXISTS continent")
session.sql("DROP TABLE IF EXISTS ocean")
session.sql("DROP TABLE IF EXISTS lake")
'''

In [ ]:
# Wyświetlenie listy istniejących tj. zarejestrowanych w katalogu tabel.
session.catalog.listTables()

In [ ]:
# Warunkowe usunięcie bazy danych earth.
session.sql("DROP DATABASE IF EXISTS earth CASCADE")

# Warunkowe utworzenie bazy danych earth.
session.sql("CREATE DATABASE IF NOT EXISTS earth")

# Wskazanie bieżącej bazy danych w ramach sesji.
session.sql("USE earth")
session.catalog.currentDatabase()

## Definiowanie tabeli Delta na podstawie danych.

### Definiowanie tabeli Delta z wykorzystaniem ramki danych DF.

In [ ]:
# Utworzenie kolekcji danych.
data = [
    (1, "Afryka"),
    (2, "Ameryka Południowa"),
    (3, "Ameryka Północna"),
    (4, "Antarktyda"),
    (5, "Australia"),
    (6, "Azja"),
    (7, "Europa")
]

# Utworzenie i wyświetlenie ramki danych DF (ang. Data Frame).
df = session.createDataFrame(
    data,
    ["id", "continent"]
)
df.show()

In [ ]:
# Utworzenie tabeli Delta.
df.write.format("delta").saveAsTable("continent")

### Definiowanie tabeli Delta z wykorzystaniem kolekcji RDD oraz ramki danych DF.

In [ ]:
# Utworzenie kolekcji danych.
data = [
    (1, "Lambert–Fisher Glacier", 97000),
    (2, "Agassiz Ice Cap", 21000),
    (3, "Severny Island Ice Cap", 20000),
    (4, "Prince of Wales Icefield", 19000),
    (5, "Devon Ice Cap", 14000),
    (6, "Southern Patagonian Ice Field", 13000),
    (7, "Austfonna", 8000),
    (8, "Vatnajökull", 7700)
]

# Utworzenie rozproszonej kolekcji RDD (ang. Resilient Distributed Datasets).
rdd = session.sparkContext.parallelize(data, 20)

# Utworzenie i wyświetlenie ramki danych DF (ang. Data Frame).
columns = ["id", "glacier", "area"]
df = rdd.toDF(columns)
df.show()

In [81]:
# Utworzenie tabeli Delta.
df.write.format("delta").saveAsTable("glacier")

### Definiowanie tabeli Delta z wykorzystaniem SQL.

In [ ]:
# Utworzenie tabeli Delta z wykorzystaniem SQL.
session.sql("""
    CREATE OR REPLACE TABLE peak (
        id integer,
        peak string,
        height integer
    ) USING delta
""")

# Dodanie danych.
session.sql("""
  INSERT INTO peak VALUES
      (1, 'Aconcagua', 6961),
      (2, 'Denali', 6190),
      (3, 'K2', 8611),
      (4, 'Lhotse', 8516),
      (5, 'Vinson', 4892)
""")

### Definiowanie tabeli Delta na podstawie danych tabeli w formacie Parquet.

In [ ]:
# Utworzenie kolekcji danych.
data = [
    (1, "Ocean Atlantycki"),
    (2, "Ocean Arktyczny"),
    (3, "Ocean Indyjski"),
    (4, "Ocean Południowy"),
    (5, "Ocean Spokojny")
]

# Utworzenie rozproszonej kolekcji RDD (ang. Resilient Distributed Datasets).
rdd = session.sparkContext.parallelize(data, 20)

# Utworzenie i wyświetlenie ramki danych DF (ang. Data Frame).
columns = ["id", "ocean"]
df = rdd.toDF(columns)
df.show()

In [112]:
# Utworzenie tabeli Parquet tj. tabeli w formacie Parquet.
# df.write.format("parquet").saveAsTable("ocean")

from pathlib import Path

delta_path = (
    Path.cwd()
    / "data"
    / "ocean1"
)

df.write.format("parquet").save(delta_path.resolve().as_uri())

In [ ]:
# Sprawdzenie formatu zapisu tabeli ocean.
from pathlib import Path

delta_path = (
    Path.cwd()
    / "spark-warehouse"
    / "earth.db"
    / "ocean"
)

delta_uri = delta_path.resolve().as_uri()
DeltaTable.isDeltaTable(session, delta_uri)


In [ ]:
print("Current directory:", __import__("os").getcwd())
print("Spark warehouse:", session.conf.get("spark.sql.warehouse.dir"))

In [ ]:
from delta.tables import DeltaTable

# Utworzenie tabeli Delta poprzez konwersję z formatu Parquet.

delta_path = (
    Path.cwd()
    / "data"
    / "ocean1"
)

parquet_path = (
#    "file:///home/tp/Documents/%23PROJECT%23/"
#    "PycharmProjects/BD.DP_BD.WH/"
#    "Laboratory_A.02_Schemas.Creation/Amudsen_1.0.0/"
#    "spark-warehouse/earth.db/ocean"
    "data/ocean1"
)

delta_table = DeltaTable.convertToDelta(
    session,
    f"parquet.`{delta_path}`"
)

In [ ]:
# Wyświetlenie kompletnych szczegółów tabeli.
session.sql(
    """
    DESCRIBE DETAIL ocean
    """
).show(truncate=False)


### **Definiowanie tabeli Lake na podstawie danych z pliku CSV.**

In [ ]:
# Utworzenie i wyświetlenie ramki danych.
# /Volumes/workspace/dataset/amudsen/continents.csv
df = session.read \
    .option("header", True) \
    .csv("/home/tpotempa/Documents/#PROJECT#/PycharmProjects/BD.DP_BD.WH/Laboratory_A.02_Schemas.Creation/Amudsen_1.0.0/data/")
    #.csv("dbfs:/Volumes/workspace/dataset/amudsen2/")
df.show()

In [ ]:
# Utworzenie tabeli Delta.
df.write.format("delta").saveAsTable("lake")

In [ ]:
# Sprawdzenie podstawowych informacji o tabeli, w tym typu tabeli.
description = session.sql(
    """
    SHOW TABLE EXTENDED IN earth LIKE 'lake'
    """
)
display(description, truncate=False)

## **Definiowanie tabeli Delta na podstawie definicji.**

### **Definiowanie tabeli Delta Lake z wykorzystaniem SQL.**

In [ ]:
# %sql
# Warunkowe usunięcie tabeli.
session.sql("DROP TABLE IF EXISTS ocean")

# Utworzenie tabeli Delta z wykorzystaniem SQL.
session.sql("""
    CREATE TABLE ocean (
        id INTEGER,
        ocean STRING
    )
    USING DELTA
    COMMENT 'Tabela przechowująca dane o oceanach.'
    """
)

### Wyświetlanie informacji o tabeli

In [ ]:
# Wyświetlenie kompletnych szczegółów tabeli.
session.sql(
    """
    DESCRIBE DETAIL peak
    """
).show(truncate=False)

# Wyświetlenie pojedynczej informacji FORMAT o tabeli.
session.sql(
    """
    DESCRIBE DETAIL peak
    """
).select("format").show(truncate=False)

# Wyświetlenie pojedynczej informacji LOCATION o tabeli.
session.sql(
    """
    DESCRIBE DETAIL peak
    """
).select("location").show(truncate=False)

In [ ]:
# Sprawdzenie podstawowych informacji o tabeli, w tym typu tabeli.
session.sql(
    """
    SHOW TABLE EXTENDED IN earth LIKE 'continent'
    """
).show(truncate=False)

### Wyświetlanie danych i optymalizacja

In [ ]:
# Wyświetlenie danych tabeli Delta.
session.table("continent").orderBy("id").show()

In [37]:
# Wyświetlenie danych tabeli Delta z wykorzystaniem SQL.
session.sql("SELECT * FROM continent ORDER BY 1").show()

+---+--------------------+-----+
| id|           continent|   _3|
+---+--------------------+-----+
|  1|Lambert–Fisher Gl...|97000|
|  2|     Agassiz Ice Cap|21000|
|  3|Severny Island Ic...|20000|
|  4|Prince of Wales I...|19000|
|  5|       Devon Ice Cap|14000|
|  6|Southern Patagoni...|13000|
|  7|           Austfonna| 8000|
|  8|         Vatnajökull| 7700|
+---+--------------------+-----+



In [ ]:
# Optymalizacja rozmiaru plików tabeli.
dt = DeltaTable.forName(session, "continent")
dt.optimize().executeCompaction()